# Playground_Ben — All Experiments

**Ben Forbes · C247 Final Project**

This notebook runs every experiment from `Playground_Ben` end-to-end:

| # | Experiment | Script / command |
|---|---|---|
| 2 | Baseline CNN (16 ch, 2000 Hz) | `emg2qwerty.train` |
| 3 | Channel Ablation (8 / 4 / 2 ch) | `emg2qwerty.train + channel_stride*.yaml` |
| 4 | Temporal Downsampling Ablation (1000–125 Hz) | `emg2qwerty.train + temporal_downsample_*.yaml` |
| 5 | Latent AE CNN | `train_latent_cnn.py` |
| 6 | Reconstructed EMG CNN | `train_recons_cnn.py` |
| 7 | Biophysics Pipeline CNN | `train_biophysics_cnn.py` |
| 8 | Hyperparameter Tuning (Latent CNN) | `hyperparam_tuner_cnn.py` |
| 8 | Hyperparameter Tuning (Biophysics CNN) | `hyperparam_tuner_raw_cnn.py` |
| 9 | Results Analysis & Plots | inline |

**Prerequisites:**
```bash
source /home/benforbes/emg2qwerty/venv/bin/activate
pip install -e .
# Then launch from repo root:
cd ~/C247_mumbikaijonathanben && jupyter notebook
```

---
## 0. Setup

In [ ]:
import os, sys, subprocess, glob, time
from pathlib import Path

# Ensure we run from the repo root regardless of where Jupyter was launched
REPO_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '..', '..'))).resolve()
if REPO_ROOT.name != 'C247_mumbikaijonathanben':
    # Fallback: walk up until we find the repo root
    for p in Path(os.getcwd()).parents:
        if (p / 'emg2qwerty').is_dir() and (p / 'Playground_Ben').is_dir():
            REPO_ROOT = p
            break

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PLAYGROUND = REPO_ROOT / 'Playground_Ben'
print(f'Repo root  : {REPO_ROOT}')
print(f'Playground : {PLAYGROUND}')

---
## 1. Patch `emg2qwerty` with Playground_Ben Modifications

Two source files are overwritten:

- **`lightning.py`** — `TDSConvCTCModule` derives electrode channel count dynamically
  from `in_features` (needed for channel ablation).
- **`transforms.py`** — adds `ChannelSelect`, `ChannelSubset`, and `TemporalDownsample` transforms.

In [ ]:
import shutil

shutil.copy(PLAYGROUND / 'emg2qwerty/transforms.py', REPO_ROOT / 'emg2qwerty/transforms.py')
shutil.copy(PLAYGROUND / 'emg2qwerty/lightning.py',  REPO_ROOT / 'emg2qwerty/lightning.py')
print('Patched: emg2qwerty/transforms.py')
print('Patched: emg2qwerty/lightning.py')

# Copy ablation configs into the main config tree so Hydra can find them
for cfg in (PLAYGROUND / 'config/transforms').glob('*.yaml'):
    dest = REPO_ROOT / 'config/transforms' / cfg.name
    shutil.copy(cfg, dest)
    print(f'Copied config: {cfg.name}')

---
## 2. Baseline CNN — 16 ch/hand, 2000 Hz, Log-Spectrogram

The unmodified `TDSConvCTC` model trained on the full 16-channel, 2000 Hz pipeline.
No Hydra transforms override is needed — this is the default configuration.

Reported results: **Val CER 18.52% · Test CER 22.28%** (training ~3 h 51 m)

In [ ]:
cmd_baseline = [
    sys.executable, '-m', 'emg2qwerty.train',
    'user=single_user',
    # No transforms override → default log_spectrogram (16 ch, 2000 Hz)
]
print('Command:', ' '.join(cmd_baseline))
result = subprocess.run(cmd_baseline, cwd=str(REPO_ROOT))
print(f'Exit code: {result.returncode}')

### 2.1 Load & plot baseline training curves

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator, SCALARS
import numpy as np
import matplotlib.pyplot as plt

UCLA = {'blue': '#2774AE', 'gold': '#FFD100', 'dark_blue': '#003B5C', 'dark_gold': '#FFB300'}


def load_tb_scalars(log_dir: Path) -> dict:
    """Load all TensorBoard scalar events from a Lightning log directory."""
    tb_dir = log_dir / 'lightning_logs' / 'version_0'
    scalars: dict = {}
    for ef in sorted(tb_dir.glob('events.out.tfevents.*')):
        ea = EventAccumulator(str(ef), size_guidance={SCALARS: 0})
        ea.Reload()
        for tag in ea.Tags().get('scalars', []):
            scalars.setdefault(tag, []).extend(ea.Scalars(tag))
    return scalars


def epoch_series(scalars, tag):
    step_to_epoch = {e.step: int(e.value) for e in scalars.get('epoch', [])}
    pairs = [(step_to_epoch[e.step], e.value)
             for e in scalars.get(tag, []) if e.step in step_to_epoch]
    if not pairs:
        return np.array([]), np.array([])
    pairs.sort()
    eps, vals = zip(*pairs)
    return np.array(eps), np.array(vals)


def plot_training_curves(scalars, title=''):
    val_ep,    val_cer    = epoch_series(scalars, 'val/CER')
    loss_ep,   val_loss   = epoch_series(scalars, 'val/loss')
    train_ep,  train_loss = epoch_series(scalars, 'train/loss')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(title, fontsize=13)

    axes[0].plot(val_ep, val_cer, color=UCLA['blue'], lw=2, label='Val CER')
    axes[0].set(xlabel='Epoch', ylabel='CER (%)', title='Validation CER')
    axes[0].set_ylim(bottom=0); axes[0].legend(); axes[0].grid(True, alpha=0.25, ls='--')

    axes[1].plot(train_ep, train_loss, color=UCLA['dark_blue'], lw=1.5, alpha=0.8, label='Train loss')
    axes[1].plot(loss_ep,  val_loss,   color=UCLA['gold'],      lw=2,   label='Val loss')
    axes[1].set(xlabel='Epoch', ylabel='CTC Loss', title='Train vs Val Loss')
    axes[1].set_ylim(bottom=0); axes[1].legend(); axes[1].grid(True, alpha=0.25, ls='--')

    plt.tight_layout()
    plt.show()
    return fig


# Auto-detect most recent run
logs = sorted((REPO_ROOT / 'logs').glob('*/*'), reverse=True)
baseline_log = next(
    d for d in logs if (d / 'lightning_logs' / 'version_0').exists()
)
print(f'Latest run: {baseline_log}')

scalars_baseline = load_tb_scalars(baseline_log)
plot_training_curves(scalars_baseline, 'CNN Baseline — 16 ch/hand, 2000 Hz')

In [ ]:
def last_value(scalars, tag):
    events = scalars.get(tag, [])
    return events[-1].value if events else float('nan')


wall_times = [e.wall_time for evts in scalars_baseline.values() for e in evts]
training_hrs = (max(wall_times) - min(wall_times)) / 3600 if wall_times else 0

print('=' * 40)
print('  CNN Baseline Results')
print('=' * 40)
print(f"  Val  CER  : {last_value(scalars_baseline, 'val/CER'):.2f}%")
print(f"  Test CER  : {last_value(scalars_baseline, 'test/CER'):.2f}%")
print(f"  Val  Loss : {last_value(scalars_baseline, 'val/loss'):.4f}")
print(f"  Train Loss: {last_value(scalars_baseline, 'train/loss'):.4f}")
print(f'  Training  : {training_hrs:.1f} hrs')
print('=' * 40)

---
## 3. Channel Ablation Study

Investigates how CTC decoding degrades as fewer sEMG electrode channels are used per hand.
The `ChannelSelect` transform (added to `transforms.py`) selects every Nth channel so that
the spatial coverage remains uniform even with fewer electrodes.

| Config | Channels/hand | `in_features` | Val CER | Test CER |
|---|---|---|---|---|
| baseline (no override) | 16 | 528 | 18.52% | 22.28% |
| `channel_stride2` | 8 | 264 | 18.65% | 23.30% |
| `channel_stride4` | 4 | 132 | 24.88% | 27.12% |
| `channel_stride8` | 2 | 66 | 40.85% | 45.00% |

The patched `lightning.py` reads `in_features` from the config automatically, so no extra
`model.in_features` override is needed.

In [ ]:
# 3.1 — 8 channels/hand (every-other electrode)
cmd_ch8 = [
    sys.executable, '-m', 'emg2qwerty.train',
    '+transforms=channel_stride2',
    'model.in_features=264',
    'user=single_user',
]
print('8 ch/hand command:', ' '.join(cmd_ch8))
result_ch8 = subprocess.run(cmd_ch8, cwd=str(REPO_ROOT))
print(f'Exit code: {result_ch8.returncode}')

In [ ]:
# 3.2 — 4 channels/hand (every 4th electrode)
cmd_ch4 = [
    sys.executable, '-m', 'emg2qwerty.train',
    '+transforms=channel_stride4',
    'model.in_features=132',
    'user=single_user',
]
print('4 ch/hand command:', ' '.join(cmd_ch4))
result_ch4 = subprocess.run(cmd_ch4, cwd=str(REPO_ROOT))
print(f'Exit code: {result_ch4.returncode}')

In [ ]:
# 3.3 — 2 channels/hand (every 8th electrode)
cmd_ch2 = [
    sys.executable, '-m', 'emg2qwerty.train',
    '+transforms=channel_stride8',
    'model.in_features=66',
    'user=single_user',
]
print('2 ch/hand command:', ' '.join(cmd_ch2))
result_ch2 = subprocess.run(cmd_ch2, cwd=str(REPO_ROOT))
print(f'Exit code: {result_ch2.returncode}')

### 3.4 Plot channel ablation results

In [ ]:
import pandas as pd

ch_csv = PLAYGROUND / 'results' / 'channel_ablation_table.csv'
df_ch = pd.read_csv(ch_csv).sort_values('channels_per_hand')
print(df_ch.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Channel Ablation — CER vs. Channels per Hand', fontsize=13)

axes[0].plot(df_ch['channels_per_hand'], df_ch['val_cer_pct'],
             'o-', color=UCLA['blue'], lw=2, label='Val CER')
axes[0].plot(df_ch['channels_per_hand'], df_ch['test_cer_pct'],
             's--', color=UCLA['dark_gold'], lw=2, label='Test CER')
axes[0].set(xlabel='Channels per hand', ylabel='CER (%)', title='CER vs Channels')
axes[0].set_xticks(df_ch['channels_per_hand'])
axes[0].legend(); axes[0].grid(True, alpha=0.25, ls='--')

axes[1].bar(df_ch['channels_per_hand'].astype(str),
            df_ch['training_time_sec'] / 3600,
            color=UCLA['blue'], edgecolor='white', width=0.5)
axes[1].set(xlabel='Channels per hand', ylabel='Training time (hrs)',
            title='Training Time vs Channels')
axes[1].grid(True, alpha=0.25, ls='--', axis='y')

plt.tight_layout()
plt.savefig(PLAYGROUND / 'plots' / 'channel_ablation_notebook.png', dpi=150)
plt.show()

---
## 4. Temporal Downsampling Ablation

Investigates CTC performance as effective sample rate is reduced by integer factors via
the `TemporalDownsample(factor)` transform (added to `transforms.py`). The transform
applies anti-aliased resampling before `LogSpectrogram`, so `in_features` is unchanged.

| Config | Factor | Effective SR | Val CER | Test CER |
|---|---|---|---|---|
| baseline | 1 | 2000 Hz | 18.52% | 22.28% |
| `temporal_downsample_2` | 2 | 1000 Hz | 52.53% | 38.38% |
| `temporal_downsample_4` | 4 | 500 Hz | 58.42% | 46.57% |
| `temporal_downsample_8` | 8 | 250 Hz | 79.15% | 76.12% |
| `temporal_downsample_16` | 16 | 125 Hz | 99.41% | 99.98% |

In [ ]:
# 4.1 — 1000 Hz (factor=2)
cmd_td2 = [
    sys.executable, '-m', 'emg2qwerty.train',
    'transforms=temporal_downsample_2',
    'user=single_user',
]
print('1000 Hz command:', ' '.join(cmd_td2))
result_td2 = subprocess.run(cmd_td2, cwd=str(REPO_ROOT))
print(f'Exit code: {result_td2.returncode}')

In [ ]:
# 4.2 — 500 Hz (factor=4)
cmd_td4 = [
    sys.executable, '-m', 'emg2qwerty.train',
    'transforms=temporal_downsample_4',
    'user=single_user',
]
print('500 Hz command:', ' '.join(cmd_td4))
result_td4 = subprocess.run(cmd_td4, cwd=str(REPO_ROOT))
print(f'Exit code: {result_td4.returncode}')

In [ ]:
# 4.3 — 250 Hz (factor=8)
cmd_td8 = [
    sys.executable, '-m', 'emg2qwerty.train',
    'transforms=temporal_downsample_8',
    'user=single_user',
]
print('250 Hz command:', ' '.join(cmd_td8))
result_td8 = subprocess.run(cmd_td8, cwd=str(REPO_ROOT))
print(f'Exit code: {result_td8.returncode}')

In [ ]:
# 4.4 — 125 Hz (factor=16)
cmd_td16 = [
    sys.executable, '-m', 'emg2qwerty.train',
    'transforms=temporal_downsample_16',
    'user=single_user',
]
print('125 Hz command:', ' '.join(cmd_td16))
result_td16 = subprocess.run(cmd_td16, cwd=str(REPO_ROOT))
print(f'Exit code: {result_td16.returncode}')

### 4.5 Plot temporal downsampling results

In [ ]:
sr_csv = PLAYGROUND / 'results' / 'sampling_ablation_table.csv'
df_sr = pd.read_csv(sr_csv).sort_values('sample_rate_hz')
print(df_sr.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Temporal Downsampling Ablation — CER vs. Sample Rate', fontsize=13)

axes[0].plot(df_sr['sample_rate_hz'], df_sr['val_cer_pct'],
             'o-', color=UCLA['blue'], lw=2, label='Val CER')
axes[0].plot(df_sr['sample_rate_hz'], df_sr['test_cer_pct'],
             's--', color=UCLA['dark_gold'], lw=2, label='Test CER')
axes[0].invert_xaxis()
axes[0].set(xlabel='Sample rate (Hz)', ylabel='CER (%)', title='CER vs Sample Rate')
axes[0].set_xticks(df_sr['sample_rate_hz'])
axes[0].legend(); axes[0].grid(True, alpha=0.25, ls='--')

axes[1].barh(df_sr['sample_rate_hz'].astype(str),
             df_sr['training_time_sec'] / 3600,
             color=UCLA['blue'], edgecolor='white')
axes[1].set(xlabel='Training time (hrs)', ylabel='Sample rate (Hz)',
            title='Training Time vs Sample Rate')
axes[1].grid(True, alpha=0.25, ls='--', axis='x')

plt.tight_layout()
plt.savefig(PLAYGROUND / 'plots' / 'sampling_ablation_notebook.png', dpi=150)
plt.show()

---
## 5. Latent AE CNN

Trains a TDSConvCTC model on **pre-computed autoencoder latent vectors** instead of raw EMG.

- **Data:** `data/emg_latent_ae_v2.hdf5` — 27,971 frames × 1024-dim float32, at 62.5 Hz (32 ms/frame)
- **Model:** `Linear(1024 → num_features)` → `TDSConvEncoder` → CTC head
- **Loader:** `Playground_Kai.data_utils.get_latent_dataloaders`

The front-end `SpectrogramNorm + MultiBandMLP` is replaced by a single linear projection,
enabling fast experiments at a drastically lower input dimensionality.

Best checkpoint saved to: `Playground_Ben/checkpoints/best_latent_cnn.pt`

In [ ]:
# Train the Latent AE CNN with default hyperparameters
# To use tuned hyperparameters, add:
#   '--from-hyperparams', str(PLAYGROUND / 'checkpoints/best_hyperparams_cnn_latent.yaml')
cmd_latent = [
    sys.executable,
    str(PLAYGROUND / 'scripts' / 'train_latent_cnn.py'),
    '--epochs', '150',
    '--lr', '3e-4',
    '--batch-size', '32',
    '--window-length', '500',
    '--stride', '50',
]
print('Latent CNN command:', ' '.join(cmd_latent))
result_latent = subprocess.run(cmd_latent, cwd=str(REPO_ROOT))
print(f'Exit code: {result_latent.returncode}')

In [ ]:
# Quick inference check on saved best checkpoint
import torch

latent_ckpt = PLAYGROUND / 'checkpoints' / 'best_latent_cnn.pt'
if latent_ckpt.exists():
    ckpt = torch.load(latent_ckpt, map_location='cpu')
    print(f'Best Latent CNN checkpoint:')
    print(f'  Epoch    : {ckpt["epoch"]}')
    print(f'  Val CER  : {ckpt["val_cer"]:.2f}%')
else:
    print('Checkpoint not found — run training cell above first.')

---
## 6. Reconstructed EMG CNN

Trains a TDSConvCTC model on **AE-reconstructed (decoded) EMG signals** stored in
`*_recons_v3.hdf5` files.

- **Data:** `data/*_recons_v3.hdf5` — AE-decoded EMG at 62.5 Hz (16 ms/frame), shape `(T, 16)` per hand
- **Model:** `Flatten(2, 16 → 32)` → `Linear(32 → num_features)` → `TDSConvEncoder` → CTC head
- **Loader:** `Playground_Ben.recons_data_utils.get_recons_dataloaders`

Unlike the latent CNN, this pipeline operates on the decoded signal space rather than
the compressed latent space — it is closer to training on raw EMG but at 32× lower rate.

Best checkpoint saved to: `Playground_Ben/checkpoints/best_recons_cnn.pt`

In [ ]:
cmd_recons = [
    sys.executable,
    str(PLAYGROUND / 'scripts' / 'train_recons_cnn.py'),
    '--epochs', '150',
    '--lr', '1e-3',
    '--batch-size', '32',
    '--window-length', '250',
    '--stride', '250',
]
print('Recons CNN command:', ' '.join(cmd_recons))
result_recons = subprocess.run(cmd_recons, cwd=str(REPO_ROOT))
print(f'Exit code: {result_recons.returncode}')

In [ ]:
recons_ckpt = PLAYGROUND / 'checkpoints' / 'best_recons_cnn.pt'
if recons_ckpt.exists():
    ckpt = torch.load(recons_ckpt, map_location='cpu')
    print('Best Reconstructed EMG CNN checkpoint:')
    print(f'  Epoch    : {ckpt["epoch"]}')
    print(f'  Val CER  : {ckpt["val_cer"]:.2f}%')
else:
    print('Checkpoint not found — run training cell above first.')

---
## 7. Biophysics Pipeline CNN

Trains a TDSConvCTC model using the **full biophysics EMG preprocessing pipeline**
(implemented in `Playground_Kai/data_preprocess.py`):

```
ToTensor → TemporalAlignmentJitter(120) → RandomBandRotation
→ ChannelSelector (8 even-indexed channels/wrist)
→ TemporalFilter (60 Hz notch + 4th-order Butterworth 20–450 Hz)
→ Decimator (2× → 1000 Hz)
→ MelSpectrogram (n_fft=256, win=64, hop=8, 32-bin Mel 20–450 Hz, log10)
→ SpecAugment (train only)
```

- **Model:** `SpectrogramNorm` → `MultiBandRotationInvariantMLP` → `TDSConvEncoder` → CTC
- **in_features:** 8 channels × 32 Mel bins = 256
- **Window:** 8000 raw samples (4 s at 2 kHz)

Best checkpoint saved to: `Playground_Ben/checkpoints/best_biophysics_cnn.pt`

To use tuned hyperparameters, pass `--from-hyperparams` (see section 8.2).

In [ ]:
cmd_biophysics = [
    sys.executable,
    str(PLAYGROUND / 'scripts' / 'train_biophysics_cnn.py'),
    '--epochs', '150',
    '--lr', '1e-3',
    '--batch-size', '32',
    '--window-length', '8000',
    '--mlp-features', '384',
    '--block-channels', '24',
    '--num-blocks', '4',
    '--kernel-width', '32',
]
print('Biophysics CNN command:', ' '.join(cmd_biophysics))
result_biophysics = subprocess.run(cmd_biophysics, cwd=str(REPO_ROOT))
print(f'Exit code: {result_biophysics.returncode}')

In [ ]:
biophysics_ckpt = PLAYGROUND / 'checkpoints' / 'best_biophysics_cnn.pt'
if biophysics_ckpt.exists():
    ckpt = torch.load(biophysics_ckpt, map_location='cpu')
    print('Best Biophysics Pipeline CNN checkpoint:')
    print(f'  Epoch    : {ckpt["epoch"]}')
    print(f'  Val CER  : {ckpt["val_cer"]:.2f}%')
else:
    print('Checkpoint not found — run training cell above first.')

---
## 8. Hyperparameter Tuning

Two two-phase Bayesian search scripts are available:

| Script | Model | Output YAML |
|---|---|---|
| `hyperparam_tuner_cnn.py` | Latent AE CNN | `best_hyperparams_cnn_latent.yaml` |
| `hyperparam_tuner_raw_cnn.py` | Biophysics CNN | `best_hyperparams_raw_cnn.yaml` |

**Phase 1 (coarse):** 20 random trials × 10 epochs each; picks top-3 configs.  
**Phase 2 (fine):** 10 trials × 20 epochs, shrinking the search space around each top-3 config.

Search space (latent CNN):
```
lr:            log-uniform [1e-4, 1e-3]
weight_decay:  log-uniform [1e-5, 1e-2]
num_features:  choice [384, 576, 768]
block_channels: choice [16, 24, 32, 48]
kernel_width:  choice [16, 24, 32, 48]
num_blocks:    choice [2, 3, 4]
```

**Best found (latent CNN):** lr=4.36e-4, wd=1.19e-5, num_features=576, block_channels=24, num_blocks=2, kernel_width=24  
**Best found (biophysics CNN):** lr=1.27e-3, wd=1.92e-3, mlp_features=512, block_channels=32, num_blocks=3, kernel_width=24, Val CER=82.41%

In [ ]:
# 8.1 Tune Latent AE CNN hyperparameters
# This runs 20 coarse + 30 fine trials — expect 1-2 hours on a GPU
cmd_tune_latent = [
    sys.executable,
    str(PLAYGROUND / 'scripts' / 'hyperparam_tuner_cnn.py'),
    '--coarse-trials', '20',
    '--coarse-epochs', '10',
    '--fine-trials', '10',
    '--fine-epochs', '20',
]
print('Tuning Latent CNN:', ' '.join(cmd_tune_latent))
result_tune_latent = subprocess.run(cmd_tune_latent, cwd=str(REPO_ROOT))
print(f'Exit code: {result_tune_latent.returncode}')

In [ ]:
# 8.2 Tune Biophysics CNN hyperparameters
cmd_tune_raw = [
    sys.executable,
    str(PLAYGROUND / 'scripts' / 'hyperparam_tuner_raw_cnn.py'),
    '--coarse-trials', '20',
    '--coarse-epochs', '8',
    '--fine-trials', '10',
    '--fine-epochs', '15',
]
print('Tuning Biophysics CNN:', ' '.join(cmd_tune_raw))
result_tune_raw = subprocess.run(cmd_tune_raw, cwd=str(REPO_ROOT))
print(f'Exit code: {result_tune_raw.returncode}')

In [ ]:
# Display saved hyperparameter files
import yaml

for fname in ['best_hyperparams_cnn_latent.yaml', 'best_hyperparams_raw_cnn.yaml']:
    hp_path = PLAYGROUND / 'checkpoints' / fname
    if hp_path.exists():
        with open(hp_path) as f:
            hp = yaml.safe_load(f)
        print(f'\n--- {fname} ---')
        for k, v in hp.items():
            print(f'  {k}: {v}')
    else:
        print(f'{fname} not found')

In [ ]:
# 8.3 Re-train latent CNN with tuned hyperparameters
hp_latent = PLAYGROUND / 'checkpoints' / 'best_hyperparams_cnn_latent.yaml'
if hp_latent.exists():
    cmd_tuned = [
        sys.executable,
        str(PLAYGROUND / 'scripts' / 'train_latent_cnn.py'),
        '--from-hyperparams', str(hp_latent),
        '--epochs', '150',
    ]
    print('Re-training with tuned hyperparams:', ' '.join(cmd_tuned))
    subprocess.run(cmd_tuned, cwd=str(REPO_ROOT))
else:
    print('Run hyperparameter tuning first (section 8.1).')

---
## 9. Results Summary & Comparison

Load pre-computed ablation CSVs, display tables, and generate a combined comparison figure.

In [ ]:
# Load ablation result tables
df_ch = pd.read_csv(PLAYGROUND / 'results' / 'channel_ablation_table.csv')
df_sr = pd.read_csv(PLAYGROUND / 'results' / 'sampling_ablation_table.csv')

print('Channel ablation results:')
print(df_ch.to_string(index=False))
print()
print('Temporal downsampling results:')
print(df_sr.to_string(index=False))

In [ ]:
# Combined ablation comparison figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ablation Study Results — Baseline TDSConvCTC', fontsize=14, fontweight='bold')

# Channel ablation
df_ch_s = df_ch.sort_values('channels_per_hand')
x_ch = df_ch_s['channels_per_hand'].values
axes[0].plot(x_ch, df_ch_s['val_cer_pct'].values,  'o-', color=UCLA['blue'],
             lw=2.5, ms=8, label='Val CER')
axes[0].plot(x_ch, df_ch_s['test_cer_pct'].values, 's--', color=UCLA['dark_gold'],
             lw=2, ms=7, label='Test CER')
for ch, vc in zip(x_ch, df_ch_s['val_cer_pct'].values):
    axes[0].annotate(f'{vc:.1f}%', (ch, vc), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=9)
axes[0].set(xlabel='Channels per hand', ylabel='CER (%)',
            title='Channel Ablation')
axes[0].set_xticks(x_ch)
axes[0].set_ylim(0, 60)
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.25, ls='--')

# Temporal ablation
df_sr_s = df_sr.sort_values('sample_rate_hz', ascending=False)
x_sr = df_sr_s['sample_rate_hz'].values
axes[1].plot(range(len(x_sr)), df_sr_s['val_cer_pct'].values,  'o-', color=UCLA['blue'],
             lw=2.5, ms=8, label='Val CER')
axes[1].plot(range(len(x_sr)), df_sr_s['test_cer_pct'].values, 's--', color=UCLA['dark_gold'],
             lw=2, ms=7, label='Test CER')
axes[1].set_xticks(range(len(x_sr)))
axes[1].set_xticklabels([f'{hz} Hz' for hz in x_sr])
axes[1].set(xlabel='Effective sample rate', ylabel='CER (%)',
            title='Temporal Downsampling Ablation')
axes[1].set_ylim(0, 110)
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.25, ls='--')

plt.tight_layout()
plt.savefig(PLAYGROUND / 'plots' / 'ablation_combined_notebook.png', dpi=150)
plt.show()

In [ ]:
# Load and display checkpoint metadata for all three latent-data models
checkpoints = {
    'Latent AE CNN':      PLAYGROUND / 'checkpoints' / 'best_latent_cnn.pt',
    'Reconstructed EMG':  PLAYGROUND / 'checkpoints' / 'best_recons_cnn.pt',
    'Biophysics Pipeline':PLAYGROUND / 'checkpoints' / 'best_biophysics_cnn.pt',
}

print(f'{'Model':<25}  {'Epoch':>6}  {'Val CER':>8}')
print('-' * 45)
for name, path in checkpoints.items():
    if path.exists():
        ckpt = torch.load(path, map_location='cpu')
        print(f'{name:<25}  {ckpt["epoch"]:>6}  {ckpt["val_cer"]:>7.2f}%')
    else:
        print(f'{name:<25}  {'—':>6}  {"(not trained)":>8}')

In [ ]:
# Display pre-generated ablation bar plots from Playground_Ben/plots/
from IPython.display import Image, display

plots_dir = PLAYGROUND / 'plots'
plot_files = [
    'channel_ablation_bar.png',
    'sampling_ablation_bar.png',
    'ablation_combined_bar.png',
]

for pf in plot_files:
    p = plots_dir / pf
    if p.exists():
        print(f'\n{pf}')
        display(Image(filename=str(p), width=800))
    else:
        print(f'{pf} not found — run the plotting scripts to regenerate.')

---
## 10. Orchestration Scripts (Shell Alternative)

The shell scripts below run full ablation suites automatically and generate summary plots.
These are provided as reference — you can run them from a terminal instead of the cells above.

```bash
cd ~/C247_mumbikaijonathanben
source /home/benforbes/emg2qwerty/venv/bin/activate

# Full channel ablation (patches module, copies configs, trains 2/4/8 ch models)
bash Playground_Ben/scripts/run_channel_ablation.sh

# Full temporal downsampling ablation
USER_CFG=single_user bash Playground_Ben/scripts/run_temporal_ablation.sh
```

Generate individual ablation plots:
```bash
python Playground_Ben/scripts/plot_channel_ablation.py
python Playground_Ben/scripts/plot_sampling_ablation.py
python Playground_Ben/scripts/plot_ablation_bars.py
```

Per-window inference visualization:
```bash
python Playground_Ben/scripts/eval_plot.py \
    --checkpoint logs/<date>/<time>/checkpoints/<epoch>.ckpt \
    --hdf5 data/<session>.hdf5 \
    --n_examples 6 \
    --out_dir Playground_Ben/plots/eval
```

EMG signal analysis (8 diagnostic plots):
```bash
python Playground_Ben/scripts/analyze_emg.py \
    --hdf5 data/<session>.hdf5 \
    --out_dir Playground_Ben/plots/emg_analysis
```